In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from lsst.ts.wep.task import CalcZernikesTask, EstimateZernikesDanishTask, EstimateZernikesTieTask, DonutStampSelectorTask
from lsst.ts.wep.utils import convertMetadataToHistory
from lsst.daf.butler import Butler
import ipywidgets
import astropy.units as u
import matplotlib.pyplot as plt
%matplotlib widget

In [ ]:
butler = Butler(
    "LSSTCam", 
    collections="LSSTCam/runs/quickLook", 
    instrument="LSSTCam"
)

In [ ]:
calcZernikesTaskConfig = CalcZernikesTask.ConfigClass()
calcZernikesTaskConfig.estimateZernikes.retarget(EstimateZernikesDanishTask)
calcZernikesTaskConfig.donutStampSelector.retarget(DonutStampSelectorTask)

calcZernikesTaskConfig.estimateZernikes.nollIndices = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 20, 21, 22, 27, 28]
# calcZernikesTaskConfig.estimateZernikes.nollIndices = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
calcZernikesTaskConfig.estimateZernikes.startWithIntrinsic = True
calcZernikesTaskConfig.estimateZernikes.returnWfDev = False
calcZernikesTaskConfig.estimateZernikes.saveHistory = True
calcZernikesTaskConfig.estimateZernikes.lstsqKwargs = {
    "ftol": 1.0e-3,
    "xtol": 1.0e-3,
    "gtol": 1.0e-3,
}
calcZernikesTaskConfig.estimateZernikes.binning = 4
calcZernikesTaskConfig.estimateZernikes.jointFitPair=True

calcZernikesTaskConfig.donutStampSelector.maxSelect = 5
calcZernikesTaskConfig.donutStampSelector.maxFracBadPixels = 2.0e-4
calcZernikesTaskConfig.donutStampSelector.useCustomSnLimit = True
calcZernikesTaskConfig.donutStampSelector.minSignalToNoise = 100
calcZernikesTask = CalcZernikesTask(config=calcZernikesTaskConfig)

In [ ]:
tieCalcZernikesTaskConfig = CalcZernikesTask.ConfigClass()
tieCalcZernikesTaskConfig.estimateZernikes.retarget(EstimateZernikesTieTask)
tieCalcZernikesTaskConfig.donutStampSelector.retarget(DonutStampSelectorTask)

tieCalcZernikesTaskConfig.estimateZernikes.nollIndices = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 20, 21, 22, 27, 28]
# tieCalcZernikesTaskConfig.estimateZernikes.startWithIntrinsic = True
# tieCalcZernikesTaskConfig.estimateZernikes.returnWfDev = False
# tieCalcZernikesTaskConfig.estimateZernikes.saveHistory = True
# tieCalcZernikesTaskConfig.estimateZernikes.lstsqKwargs = {
#     "ftol": 1.0e-3,
#     "xtol": 1.0e-3,
#     "gtol": 1.0e-3,
# }
# tieCalcZernikesTaskConfig.estimateZernikes.binning = 4
# tieCalcZernikesTaskConfig.estimateZernikes.jointFitPair=True

tieCalcZernikesTaskConfig.donutStampSelector.maxSelect = 5
tieCalcZernikesTaskConfig.donutStampSelector.maxFracBadPixels = 2.0e-4
tieCalcZernikesTaskConfig.donutStampSelector.useCustomSnLimit = True
tieCalcZernikesTaskConfig.donutStampSelector.minSignalToNoise = 100
tieCalcZernikesTask = CalcZernikesTask(config=tieCalcZernikesTaskConfig)

In [ ]:
_day_obs = 20250426
_seq_num = 467
_binning = 2
_detector = 191
results = None
tieResults = None

In [ ]:
@ipywidgets.interact_manual(
    day_obs=ipywidgets.IntText(value=_day_obs),
    seq_num=ipywidgets.IntText(value=_seq_num),
    binning=ipywidgets.IntText(value=_binning),
    detector=ipywidgets.IntText(value=_detector),
)
def f(day_obs, seq_num, binning, detector):
    global _day_obs    
    global _seq_num    
    global _binning
    global _detector
    global results
    global tieResults
    _day_obs = day_obs
    _seq_num = seq_num
    _detector = detector
    _binning = binning

    calcZernikesTaskConfig.estimateZernikes.binning = binning
    calcZernikesTask = CalcZernikesTask(config=calcZernikesTaskConfig)    

    
    donutStampsIntra = butler.get("donutStampsIntra", day_obs=day_obs, seq_num=seq_num, detector=detector)
    donutStampsExtra = butler.get("donutStampsExtra", day_obs=day_obs, seq_num=seq_num, detector=detector)
    
    calcZernikesTask.log.setLevel("CRITICAL")
    results = calcZernikesTask.run(donutStampsIntra=donutStampsIntra, donutStampsExtra=donutStampsExtra)
    metadata = calcZernikesTask.getFullMetadata()
    history = convertMetadataToHistory(metadata["calcZernikesBaseTask:estimateZernikes"]["history"])

    tieResults = tieCalcZernikesTask.run(donutStampsIntra=donutStampsIntra, donutStampsExtra=donutStampsExtra)
    
    fig, axs = plt.subplots(ncols=6, nrows=5, figsize=(10, 8))
    for i, pair in enumerate(history.values()):
        img = pair['intra']['image']
        mod = pair['intra']['model']
        axs[i, 0].imshow(img, origin='lower')
        axs[i, 1].imshow(mod, origin='lower')
        axs[i, 2].imshow(img-mod, origin='lower')
    
        img = pair['extra']['image']
        mod = pair['extra']['model']
        axs[i, 3].imshow(img, origin='lower')
        axs[i, 4].imshow(mod, origin='lower')
        axs[i, 5].imshow(img-mod, origin='lower')
    for ax in axs.ravel():
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(f"{day_obs = }   {seq_num = }   {detector = }")
    plt.tight_layout()
    plt.show()

    zea = butler.get("zernikeEstimateAvg", day_obs=day_obs, seq_num=seq_num, detector=detector)
    for j, zk, zea_, tiezk in zip(calcZernikesTaskConfig.estimateZernikes.nollIndices, results.outputZernikesAvg.T, zea[0], tieResults.outputZernikesAvg.T):
        print(f"Z{j:02d}  {zk[0]*1e3:10.4f} nm   {zea_*1e3:10.4f} nm {tiezk[0]*1e3:10.4f} nm")